# From structured grid to Tecplot
## Данный скрипт выполняет обработку данных из structured grid в формат Tecplot

In [1]:
import pandas as pd
import os

In [4]:
# папка с файлами
folder = "data/Combustion"

files = {
    "U": "combustionU.txt",
    "V": "combustionV.txt",
    # "T": "combustionT.txt",
    "O2": "combustionO2.txt",
    "CO2": "combustionCO2.txt",
    "CH4": "combustionCH4.txt",
    "CO": "combustionCO.txt",
}

data_vars = {}

# читаем первый файл, чтобы получить X и Y
first = list(files.values())[0]
df = pd.read_csv(os.path.join(folder, first), sep=r"\s+", header=None)

x = df.iloc[0, 1:].values.astype(float)
y = df.iloc[1:, 0].values.astype(float)

NX = len(x)
NY = len(y)

# читаем все поля
for var_name, filename in files.items():
    df = pd.read_csv(os.path.join(folder, filename), sep=r"\s+", header=None)
    Z = df.iloc[1:, 1:].values.astype(float)
    data_vars[var_name] = Z

# запись в Tecplot
with open("combined.dat", "w") as f:
    # VARIABLES
    var_list = ['"X"', '"Y"'] + [f'"{v}"' for v in files.keys()]
    f.write("VARIABLES = " + ", ".join(var_list) + "\n")

    # ZONE
    f.write(f"ZONE I={NX}, J={NY}, F=POINT\n")

    # запись данных
    for j in range(NY):
        for i in range(NX):
            line = [x[i], y[j]]

            for var in files.keys():
                line.append(data_vars[var][j, i])

            f.write(" ".join(map(str, line)) + "\n")